In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
import requests
import os

load_dotenv()
api_key = os.getenv('apptweak_api_key')

headers = {'accept': 'application/json',
           'X-Apptweak-Key': api_key}


In [ ]:
import json
from pathlib import Path

def save_json(data, path: str, filename: str):
    """
    Save Python object as JSON.
    - Creates folders if they don't exist
    - Saves with indent=2
    """
    path = Path(path)
    path.mkdir(parents = True, exist_ok = True)

    file_path = path / filename

    with open(file_path, 'w', encoding = 'utf-8') as f:
        json.dump(data, f, indent = 2, ensure_ascii = False)

In [32]:
def request_data(url: str, headers: dict, params: dict, path: str) -> dict:
    """
    """
    response = requests.get(url, headers = headers, params = params)

    if response.status_code != 200:
        print(f"Error Code: {response.status_code} for {url}")
        print(response.text[:300])
        return None

    data = response.json()
    save_json(data, f"{path}.json")

    return data

In [48]:
def build_base_params(apps_by_device: dict, start_date: str, end_date: str, countries: list) -> dict:
    base_params_dict = {}

    for country in countries:
        for device, apps in apps_by_device.items():

            key = f"{country}_{device}"
        
            params = {'apps': ','.join(apps),
                      'start_date': start_date,
                      'end_date': end_date,
                      'country': country,
                      'device': device,}

            base_params_dict[key] = params

    return base_params_dict

def build_metrics_params(base_params: dict, metrics: list) -> dict:
    new_params = {}

    for key, params in base_params.items():
        new_params[key] = params.copy()
        new_params[key]['metrics'] = ','.join(metrics)

    return new_params

def build_keywords_params(base_params: dict, keywords: list, keywords_metrics: list) -> dict:
    new_params = {}

    for key, params in base_params.items():
        new_params[key] = params.copy()
        new_params[key]['keywords'] = ','.join(keywords)
        new_params[key]['metrics'] = ','.join(keywords_metrics)

    return new_params

def build_category_params(start_date: str, end_date: str, countries: list, categories_map: dict, category_metrics: list) -> dict:
    category_params = {}
    categories = {}

    for _, platform_map in categories_map.items():
        for device, value in platform_map.items():

            if device not in categories:
                categories[device] = []

            categories[device].append(value)

    for device in categories.keys():
        for metric in category_metrics:

            params = {
            'metric': metric,
            'categories': ','.join(categories[device]),
            'countries': ','.join(countries),
            'start_date': start_date,
            'end_date': end_date,
            'type': 'free',
            'device': device
            }

            category_params[f"{device}_{metric}"] = params

    return category_params

In [47]:
"""
Configurations:


"""
apps = ['Tinder', 'Bumble', 'Hinge', 'Coffee Meets Bagel']

apps_by_device = {
    'android': ['com.tinder', 'com.bumble.app', 'co.hinge.app', 'com.coffeemeetsbagel'],
    'iphone': ['547702041','930441707', '595287172', '6502307144']
}
start_date = '2026-03-01'
end_date = '2026-03-31'
countries = ['us', 'de', 'jp']

metrics = ['downloads', 'revenues' , 'ratings']

keywords = ['dating app', 'online dating', 'meet people', 'find love', 'chat app']
keywords_metrics = ['rank', 'installs']

categories_map = {
    'social' : {'android' : 'SOCIAL', 'iphone': '6005'},
    'lifestyle': {'android': 'LIFESTYLE', 'iphone': '6012'},
    'entertainment': {'android': 'ENTERTAINMENT', 'iphone': '6016'},
    'dating': {'android': 'DATING'},
}
category_metrics = ['downloads', 'revenues']

In [23]:
base_params = build_base_params(apps_by_device, start_date, end_date, countries)

print(json.dumps(base_params, indent = 2))

{
  "us_android": {
    "apps": "com.tinder,com.bumble.app,co.hinge.app,com.coffeemeetsbagel",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "country": "us",
    "device": "android"
  },
  "us_iphone": {
    "apps": "547702041,930441707,595287172,6502307144",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "country": "us",
    "device": "iphone"
  },
  "de_android": {
    "apps": "com.tinder,com.bumble.app,co.hinge.app,com.coffeemeetsbagel",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "country": "de",
    "device": "android"
  },
  "de_iphone": {
    "apps": "547702041,930441707,595287172,6502307144",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "country": "de",
    "device": "iphone"
  },
  "jp_android": {
    "apps": "com.tinder,com.bumble.app,co.hinge.app,com.coffeemeetsbagel",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "country": "jp",
    "device": "android"
  },
  "jp_iphone": 

In [24]:
metrics_params = build_metrics_params(base_params, metrics)

print(json.dumps(metrics_params, indent = 2))

{
  "us_android": {
    "apps": "com.tinder,com.bumble.app,co.hinge.app,com.coffeemeetsbagel",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "country": "us",
    "device": "android",
    "metrics": "downloads,revenues,ratings"
  },
  "us_iphone": {
    "apps": "547702041,930441707,595287172,6502307144",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "country": "us",
    "device": "iphone",
    "metrics": "downloads,revenues,ratings"
  },
  "de_android": {
    "apps": "com.tinder,com.bumble.app,co.hinge.app,com.coffeemeetsbagel",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "country": "de",
    "device": "android",
    "metrics": "downloads,revenues,ratings"
  },
  "de_iphone": {
    "apps": "547702041,930441707,595287172,6502307144",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "country": "de",
    "device": "iphone",
    "metrics": "downloads,revenues,ratings"
  },
  "jp_android": {
    "apps": "com.tinde

In [50]:
keywords_params = build_keywords_params(base_params, keywords, keywords_metrics)

print(json.dumps(keywords_params, indent = 2))

{
  "us_android": {
    "apps": "com.tinder,com.bumble.app,co.hinge.app,com.coffeemeetsbagel",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "country": "us",
    "device": "android",
    "keywords": "dating app,online dating,meet people,find love,chat app",
    "metrics": "rank,installs"
  },
  "us_iphone": {
    "apps": "547702041,930441707,595287172,6502307144",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "country": "us",
    "device": "iphone",
    "keywords": "dating app,online dating,meet people,find love,chat app",
    "metrics": "rank,installs"
  },
  "de_android": {
    "apps": "com.tinder,com.bumble.app,co.hinge.app,com.coffeemeetsbagel",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "country": "de",
    "device": "android",
    "keywords": "dating app,online dating,meet people,find love,chat app",
    "metrics": "rank,installs"
  },
  "de_iphone": {
    "apps": "547702041,930441707,595287172,6502307144",
    "start

In [26]:
category_params = build_category_params(start_date, end_date, countries, categories_map, category_metrics)

print(json.dumps(category_params, indent = 2))

{
  "android_downloads": {
    "metric": "downloads",
    "categories": "SOCIAL,LIFESTYLE,ENTERTAINMENT,DATING",
    "countries": "us,de,jp",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "type": "free",
    "device": "android"
  },
  "android_revenues": {
    "metric": "revenues",
    "categories": "SOCIAL,LIFESTYLE,ENTERTAINMENT,DATING",
    "countries": "us,de,jp",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "type": "free",
    "device": "android"
  },
  "iphone_downloads": {
    "metric": "downloads",
    "categories": "6005,6012,6016",
    "countries": "us,de,jp",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "type": "free",
    "device": "iphone"
  },
  "iphone_revenues": {
    "metric": "revenues",
    "categories": "6005,6012,6016",
    "countries": "us,de,jp",
    "start_date": "2026-03-01",
    "end_date": "2026-03-31",
    "type": "free",
    "device": "iphone"
  }
}


In [57]:
metrics_url = 'https://public-api.apptweak.com/api/public/store/apps/metrics/history.json'
keywords_ranking_url = 'https://public-api.apptweak.com/api/public/store/apps/keywords-rankings/history.json'
category_ranking_url = 'https://public-api.apptweak.com/api/public/store/apps/category-rankings/history.json'
categories_url = 'https://public-api.apptweak.com/api/public/store/categories/metrics'


In [ ]:
metrics_output = {}

for key in metrics_params.keys():

    print(f"Fetching data for {key}...")

    data = request_data(keywords_ranking_url, headers, metrics_params[key])

    if data:
        metrics_output[key] = data
        save_json(data, '../data/raw/metrics/single', f"{key}_metrics.json")
    else:
        print(f"Request for {key} metrics produced no results!")

save_json(metrics_output, '../data/raw/metrics/compiled', 'all_metrics.json')

Fetching data for us_android...
Fetching data for us_iphone...
Fetching data for de_android...
Fetching data for de_iphone...
Fetching data for jp_android...
Fetching data for jp_iphone...


In [ ]:
keywords_output = {}

for key in keywords_params.keys():

    print(f"Fetching data for {key}...")

    data = request_data(keywords_ranking_url, headers, keywords_params[key])

    if data:
        keywords_output[key] = data
        save_json(data, '../data/raw/keywords/single', f"{key}_keywords.json")
    else:
        print(f"Request for {key} metrics produced no results!")

save_json(keywords_output, '../data/raw/keywords/compiled', 'all_keywords.json')

Fetching data for us_android...
Fetching data for us_iphone...
Fetching data for de_android...
Fetching data for de_iphone...
Fetching data for jp_android...
Fetching data for jp_iphone...


In [59]:
category_ranking_output = {}

for key in base_params.keys():

    print(f"Fetching data for {key}...")

    data = request_data(category_ranking_url, headers, base_params[key])

    if data:
        category_ranking_output[key] = data
        save_json(data, '../data/raw/category_ranking/single', f"{key}_category_ranking.json")
    else:
        print(f"Request for {key} metrics produced no results!")

save_json(category_ranking_output, '../data/raw/category_ranking/compiled', 'all_category_rankings.json')

Fetching data for us_android...
Fetching data for us_iphone...
Fetching data for de_android...
Fetching data for de_iphone...
Fetching data for jp_android...
Fetching data for jp_iphone...


In [58]:
categories_output = {}

for key in category_params.keys():

    print(f"Fetching data for {key}...")

    data = request_data(categories_url, headers, category_params[key])

    if data:
        categories_output[key] = data
        save_json(data, '../data/raw/category_metrics/single', f"{key}_category_metrics.json")
    else:
        print(f"Request for {key} metrics produced no results!")

save_json(categories_output, '../data/raw/category_metrics/compiled', f"all_category_metrics.json")

Fetching data for android_downloads...
Fetching data for android_revenues...
Fetching data for iphone_downloads...
Fetching data for iphone_revenues...


In [ ]:
app_map = {
    'com.tinder': 'tinder',
    'com.bumble.app': 'bumble',
    'co.hinge.app': 'hinge',
    'com.coffeemeetsbagel': 'cmb',
    '6502307144': 'tinder',
    '930441707': 'bumble',
    '595287172': 'hinge',
    '547702041': 'cmb'
}

In [121]:
rows = []

for run_key, payload in metrics_output.items():

    country, device = run_key.split('_')

    result = payload['result']

    for app, metrics in result.items():

        for metric_name, series in metrics.items():

            if metric_name == 'ratings':
                continue
            
            for point in series:

                rows.append({
                    'date': point['date'],
                    'app': app,
                    'country': country,
                    'device': device,
                    'metric': metric_name,
                    'value': point['value'],
                    'precision': point.get('precision')
                })

metrics_df = pd.DataFrame(rows)
metrics_df['app'] = metrics_df['app'].replace(app_map)

path = Path('../data/processed/dataframes')
path.mkdir(parents = True, exist_ok = True)

metrics_df.to_csv(f"{path}/metrics_df.csv", index = False)

print(metrics_df.head())

         date     app country   device     metric    value  precision
0  2026-03-01  tinder      us  android  downloads  12245.0        1.0
1  2026-03-02  tinder      us  android  downloads  12138.0        1.0
2  2026-03-03  tinder      us  android  downloads  12475.0        1.0
3  2026-03-04  tinder      us  android  downloads  10050.0        1.0
4  2026-03-05  tinder      us  android  downloads  10604.0        1.0


In [134]:
rows = []

for run_key, payload in metrics_output.items():

    country, device = run_key.split('_')

    result = payload['result']

    for app, metrics in result.items():

        if 'ratings' not in metrics:
            continue

        series = metrics['ratings']

        for point in series:

            breakdown = point.get('breakdown', {})

            if breakdown:

                rows.append({
                    'date': point['date'],
                    'app': app,
                    'country': country,
                    'device': device,

                    'avg_rating': breakdown.get('avg'),
                    'rating_1': breakdown.get('1'),
                    'rating_2': breakdown.get('2'),
                    'rating_3': breakdown.get('3'),
                    'rating_4': breakdown.get('4'),
                    'rating_5': breakdown.get('5'),
                    'rating_total': breakdown.get('total')
                    })
            else:
                rows.append({
                    'date': point['date'],
                    'app': app,
                    'country': country,
                    'device': device,

                    'avg_rating': None,
                    'rating_1': None,
                    'rating_2': None,
                    'rating_3': None,
                    'rating_4': None,
                    'rating_5': None,
                    'rating_total': None,
                    })                

ratings_df = pd.DataFrame(rows)
ratings_df['app'] = ratings_df['app'].replace(app_map)

ratings_df.to_csv(f"{path}/ratings_df.csv", index = False)

print(ratings_df.head())

         date     app country   device  avg_rating   rating_1  rating_2  \
0  2026-03-01  tinder      us  android       3.763  1821661.0  372068.0   
1  2026-03-02  tinder      us  android       3.764  1819873.0  371875.0   
2  2026-03-03  tinder      us  android       3.765  1819468.0  371330.0   
3  2026-03-04  tinder      us  android       3.765  1820551.0  371609.0   
4  2026-03-05  tinder      us  android       3.765  1820325.0  371345.0   

   rating_3  rating_4   rating_5  rating_total  
0  715618.0  929363.0  4862714.0     8701424.0  
1  715181.0  930131.0  4865873.0     8702933.0  
2  714637.0  931278.0  4868415.0     8705128.0  
3  713393.0  932355.0  4868688.0     8706596.0  
4  712672.0  933027.0  4871537.0     8708906.0  


In [113]:
rows = []

for run_key, payload in keywords_output.items():
    
    country, device = run_key.split('_')

    result = payload['result']

    for app, keywords in result.items():

        for keyword, metrics in keywords.items():

            for metric_name, series in metrics.items():

                for point in series:

                    rows.append({
                        'date': point['date'],
                        'app': app,
                        'country': country,
                        'device': device,
                        'keyword': keyword,
                        'metric': metric_name,
                        'value': point['value'],
                        'effective_value': point.get('effective_value')
                    })

keywords_df = pd.DataFrame(rows)
keywords_df['app'] = keywords_df['app'].replace(app_map)

keywords_df.to_csv(f"{path}/keywords_df.csv", index = False)

print(keywords_df.head())

         date     app country   device     keyword metric  value  \
0  2026-03-01  tinder      us  android  dating app   rank    1.0   
1  2026-03-02  tinder      us  android  dating app   rank    1.0   
2  2026-03-03  tinder      us  android  dating app   rank    1.0   
3  2026-03-04  tinder      us  android  dating app   rank    1.0   
4  2026-03-05  tinder      us  android  dating app   rank    2.0   

   effective_value  
0              1.0  
1              1.0  
2              1.0  
3              1.0  
4              2.0  


In [112]:
rows = []

for run_key, payload in category_ranking_output.items():

    country, device = run_key.split('_')

    result = payload['result']

    for app, app_data in result.items():

        rankings = app_data['rankings']

        for r in rankings:

            for v in r['value']:

                rows.append({
                    'date': v['fetch_date'],
                    'app': app,
                    'country': country,
                    'device': device,
                    'category_name': v['category_name'].lower(),
                    'chart_type': v['chart_type'],
                    'rank': v['rank'],
                    'fetch_depth': v['fetch_depth']
                })
                
category_ranking_df = pd.DataFrame(rows)
category_ranking_df['app'] = category_ranking_df['app'].replace(app_map)
category_ranking_df['category_name'] = category_ranking_df['category_name'].replace({'category all' : 'all'})

category_ranking_df.to_csv(f"{path}/category_ranking_df.csv", index = False)

print(category_ranking_df.head())

         date     app country   device category_name chart_type  rank  \
0  2026-03-01  tinder      us  android        dating       free   1.0   
1  2026-03-01  tinder      us  android   application       free   NaN   
2  2026-03-01  tinder      us  android        dating   grossing   1.0   
3  2026-03-01  tinder      us  android           all   grossing  22.0   
4  2026-03-01  tinder      us  android   application   grossing  11.0   

   fetch_depth  
0          200  
1          200  
2          200  
3          195  
4          200  


In [116]:
rows = []

for run_key, payload in categories_output.items():

    device, metric = run_key.split('_')

    result = payload['result']

    for category, countries in result.items():

        for country, series in countries.items():

            for point in series:

                rows.append({
                    'date': point['date'],
                    'category': category.lower(),
                    'country': country,
                    'device': device,
                    'metric': metric,
                    'value': point.get(metric) 
                })

categories_df = pd.DataFrame(rows)
categories_df.to_csv(f"{path}/categories_df.csv", index = False)

print(categories_df.head())

         date   category country   device     metric     value
0  2026-03-20  lifestyle      de  android  downloads  108279.0
1  2026-03-26  lifestyle      de  android  downloads  113375.0
2  2026-03-08  lifestyle      de  android  downloads  242040.0
3  2026-03-06  lifestyle      de  android  downloads  235777.0
4  2026-03-11  lifestyle      de  android  downloads  109401.0
